# LSTM Analysis

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from sklearn.preprocessing import MinMaxScaler

2026-04-03 23:06:16.987842: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
h_df = pd.read_csv('../data/treated_historic_data.csv', parse_dates=['date'], index_col=0)
f_df = pd.read_csv('../data/treated_forecast_data.csv', parse_dates=['date'], index_col=0)

In [4]:
TARGET = 'temperature_2m_seville'
FEATURES = [f for f in h_df.columns if f not in [TARGET, 'd_type']]

X_train, y_train = h_df[FEATURES], h_df[TARGET]
X_test, y_test = f_df[FEATURES], f_df[TARGET]

Scaling data

In [5]:
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

X_train_s = scaler_X.fit_transform(X_train)
X_test_s  = scaler_X.transform(X_test)

y_train_s = scaler_y.fit_transform(y_train.values.reshape(-1, 1))
y_test_s  = scaler_y.transform(y_test.values.reshape(-1, 1))

We need to set the data into 3D. Now it is in 2D

In [6]:
n_features = X_train_s.shape[1]

X_train_3d = X_train_s.reshape((X_train_s.shape[0], 1, n_features))
X_test_3d  = X_test_s.reshape((X_test_s.shape[0], 1, n_features))

Define the model

In [9]:
model = Sequential([
    LSTM(64, activation='relu', input_shape=(1, n_features)),
    Dense(1)
])

model.compile(optimizer='adam', loss='mse')
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_1 (LSTM)                   │ (None, 64)             │        33,280 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 33,345 (130.25 KB)

 Trainable params: 33,345 (130.25 KB)

 Non-trainable params: 0 (0.00 B)

Train it

In [10]:
history = model.fit(
    X_train_3d, y_train_s,
    epochs=50,
    batch_size=32,
    validation_split=0.1,
    verbose=1
)

Epoch 1/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 5s 56ms/step - loss: 0.1415 - val_loss: 0.1126
Epoch 2/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0592 - val_loss: 0.0355
Epoch 3/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0388 - val_loss: 0.0238
Epoch 4/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0225 - val_loss: 0.0099
Epoch 5/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0132 - val_loss: 0.0052
Epoch 6/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0074 - val_loss: 0.0043
Epoch 7/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0042 - val_loss: 0.0072
Epoch 8/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 0.0029 - val_loss: 0.0080
Epoch 9/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - loss: 0.0024 - val_loss: 0.0088
Epoch 10/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.0020 - val_loss: 0.0083
Epoch 11/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - loss: 0.0017 - val_loss: 0.0068
Epoch 12/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - loss: 0.0

Predict and unscale

In [11]:
y_pred_s = model.predict(X_test_3d)
y_pred   = scaler_y.inverse_transform(y_pred_s)
y_real   = scaler_y.inverse_transform(y_test_s)

5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step


Some metrics

In [14]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

mae  = mean_absolute_error(y_real, y_pred)
rmse = root_mean_squared_error(y_real, y_pred)
r2 = r2_score(y_real, y_pred)

print(f"MAE:  {mae:.4f} °C")
print(f"RMSE: {rmse:.4f} °C")
print(f"R2: {r2:.4f}")

MAE:  1.2240 °C
RMSE: 1.4796 °C
R2: 0.9313
